# Notebook 2ου εργαστηρίου
*Πριν προχωρήσετε, εκτελέστε τα επόμενα δύο κελιά.*

In [ ]:
interp.configureCompiler(_.settings.processArguments(List("-Wconf:cat=deprecation:s"), true))
interp.load.module(os.Path(s"${System.getProperty("user.dir")}/resource/chisel_deps.sc"))

In [ ]:
import chisel3._
import chisel3.util._
import chiseltest._
import chiseltest.RawTester.test

## 2.1 Βασικές πράξεις μεταξύ bits
Η Chisel ορίζει τελεστές για τις βασικές πράξεις μεταξύ bits ως εξής:

| Τελεστής | Πράξη μεταξύ bits |
| -------- | ----------------- |
| ~ | NOT |
| & | AND |
| ^ | XOR |
| \| | OR |


Η προτεραιότητα των τελεστών στον προηγούμενο πίνακα είναι από πάνω προς τα κάτω, με την υψηλότερη προτεραιότητα στο NOT και τη χαμηλότερη στο OR.

Οι τελεστές εφαρμόζονται σε οποιαδήποτε σήματα και έχουν ως αποτέλεσμα την κατασκευή του κυκλώματος που θα εκτελεσει τις αντίστοιχες λογικές πράξεις. Μπορείτε π.χ. να γράψετε μέσα σε ένα module:

~~~scala
io.out := ~io.a & io.b
~~~

Το προηγούμενο σημαίνει ότι το σήμα `io.out` παράγεται από τον συνδυασμό με τη λογική πράξη AND των σημάτων `NOT(io.a)` και `io.b`.

### 2.1.1 Αυτόματη προσαρμογή στο εύρος των σημάτων
Ένα μεγάλο πλεονέκτημα της Chisel είναι ότι η **ίδια έκφραση** μπορεί να κατασκευάσει πολλαπλά κυκλώματα, ανάλογα με το **εύρος** (width) των σημάτων που μετέχουν στην έκφραση. Για παράδειγμα, η έκφραση

~~~scala
io.y := io.a & io.b
~~~

θα φτιάξει το κύκλωμα `y = a AND b` (μία πύλη AND) αν τα a και b έχουν εύρος 1 bit.

Εάν όμως το εύρος των a και b είναι π.χ. 4 bits, **η ίδια έκφραση** (`io.y := io.a & io.b`) θα φτιάξει τα εξής (4 πύλες AND):
~~~
y(3) = a(3) AND b(3)
y(2) = a(2) AND b(2)
y(1) = a(1) AND b(1)
y(0) = a(0) AND b(0)
~~~

Το εύρος του αποτελέσματος των τελεστών πράξεων με bits προσαρμόζεται ως εξής:

| τελεστής | εύρος αποτελέσματος |
| --- | --- |
| ~x (NOT) | εύρος(x) |
| x & y (AND) | max(εύρος(x),εύρος(y)) |
| x ^ y (XOR) | max(εύρος(x),εύρος(y)) |
| x \| y (OR) | max(εύρος(x),εύρος(y)) |


## 2.2 Περιγραφή συμπεριφοράς κυκλωμάτων
Στην Chisel σπάνια θα χρειαστεί να περιγράψουμε τη συμπεριφορά ενός module στο χαμηλό επίπεδο των λογικών πράξεων με πύλες. Στο εργαστήριο αυτό θα δούμε **πώς μπορούμε να περιγράψουμε ένα κύκλωμα μέσα από την αναμενόμενη συμπεριφορά** του. Όπως και σε όλες τις άλλες περιπτώσεις, η Chisel θα κατασκευάσει μια περιγραφή κυκλώματος, η οποία όταν υλοποιηθεί με κάποια τεχνολογία θα έχει τη ζητούμενη συμπεριφορά.

Η περιγραφή ενός κυκλώματος μέσω της αναμενόμενης συμπεριφοράς του **είναι η προτιμώμενη μέθοδος στην Chisel**.

### 2.2.1 Παράδειγμα: ο πολυπλέκτης 4-σε-1
Ο πολυπλέκτης 4-σε-1 είναι ένα κύκλωμα με:
* Μία είσοδο ``in`` με εύρος 4 bits.
* Μία είσοδο ``sel`` με εύρος 2 bits.
* Μία έξοδο ``out`` με εύρος 1 bit.

Ανάλογα με την τιμή της εισόδου ``sel``, η έξοδος ``out`` ισούται με κάποιο από τα bits της εισόδου ``in`` σύμφωνα με τον πίνακα:

| sel |  out  |
|-----|-------|
| 0.U | in(0) |
| 1.U | in(1) |
| 2.U | in(2) |
| 3.U | in(3) |

Στο παράδειγμα που ακολουθεί περιγράφουμε τον **πολυπλέκτη 4-σε-1** με τρεις διαφορετικούς τρόπους:

1. Με πύλες (λογικές πράξεις)
2. Με περιγραφή της αναμενόμενης συμπεριφοράς μέσω της δομής `when..elsewhen..otherwise`.
2. Με περιγραφή της αναμενόμενης συμπεριφοράς μέσω της δομής `switch`.



### 2.2.2 Περιγραφή με λογικές πράξεις

Στο επόμενο σχήμα εμφανίζεται μια υλοποίηση του **πολυπλέκτη 4-σε-1** με πύλες (πράξεις με bits):

![mux4to1.png](https://mixstef.github.io/courses/comparch/labimg/mux4to1.png)

Και ακολουθεί η υλοποίηση του πολυπλέκτη 4-σε-1 με λογικές πύλες στην Chisel στο επόμενο κελί (*εκτελέστε το*):

In [ ]:
class Mux4to1 extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(4.W))
    val out = Output(UInt(1.W))
    val sel = Input(UInt(2.W))
  })
    
  val p0 = io.in(0) & ~io.sel(0) & ~io.sel(1)
  val p1 = io.in(1) & io.sel(0) & ~io.sel(1)
  val p2 = io.in(2) & ~io.sel(0) & io.sel(1)
  val p3 = io.in(3) & io.sel(0) & io.sel(1)
  io.out := p0 | p1 | p2 | p3
}

### 2.2.3 Έλεγχος ορθότητας του πολυπλέκτη 4-σε-1
Στο επόμενο κελί, ο έλεγχος ορθότητας εξετάζει για κάθε συνδυασμό του `io.sel` (εξωτερικό for από 0 έως 3), αν εμφανίζεται στην έξοδο η κατάλληλη είσοδος για κάθε πιθανή τιμή εισόδου `io.in` (εσωτερικό for από 0 έως 15). 
Xρησιμοποιούμε την υποκείμενη γλώσσα (Scala) για να αυτοματοποιήσουμε τον έλεγχο, για τις λεπτομέρειες δείτε το κείμενο μετά τον κώδικα.

*Εκτελέστε το επόμενο κελί.*

In [ ]:
try {
  test(new Mux4to1()) { c =>
    for (s <- 0 to 3) {
      val mask = 1 << s
      c.io.sel.poke(s.U)
      for (i <- 0 to 15) {
        c.io.in.poke(i.U)      
        c.io.out.expect(if ((i&mask)!=0) 1.U else 0.U)
      }
    }
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

#### Εξήγηση κώδικα
Ο κώδικας ελέγχου χρησιμοποιεί 2 loops:

Το εξωτερικό loop θέτει τη μεταβλητή `s` σε κάθε πιθανή τιμή (0 έως 3) για την είσοδο `io.sel`. Ανάλογα με το `s` παίρνει τιμή η μεταβλητή `mask` (μια δυαδική ποσότητα που έχει ένα 1 στα bits 0, 1, 2 ή 3), η οποία χρησιμοποιείται για να απομονώνουμε την τιμή της επιλεγμένης εισόδου `io.in(s)` και να ελέγχουμε αν αυτή η είσοδος ταιριάζει με την έξοδο. 

~~~scala
for (s <- 0 to 3) {
  val mask = 1 << s
  c.io.sel.poke(s.U)
    
}
~~~

Το εσωτερικό loop βάζει κάθε πιθανή τιμή (0 έως 15) στη μεταβλητή `i` για την είσοδο `io.in`. Στη συνέχεια ελέγχει αν στο `io.out` εμφανίζεται η τιμή `0.U` ή `1.U` ανάλογα με το επιλεγμένο bit του `i`.  

~~~scala
      for (i <- 0 to 15) {
        c.io.in.poke(i.U)      
        c.io.out.expect(if ((i&mask)!=0) 1.U else 0.U)
      }
~~~



### 2.2.4 Περιγραφή συμπεριφοράς με τη δομή when..elsewhen..otherwise

Ο ίδιος πολυπλέκτης 4-σε-1 με τη δομή `when..elsewhen..otherwise`.

**Προσέξτε ότι η ισότητα σημάτων στην Chisel ελέγχεται με το `===`(τριπλό ίσον)!**

*Εκτελέστε το επόμενο κελί που επανακαθορίζει το module `Mux4to1`. Στη συνέχεια τρέξτε το προηγούμενο test (ενότητα 2.2.3) για να βεβαιωθείτε ότι και αυτή η εκδοχή του πολυπλέκτη δουλεύει κατά το αναμενόμενο.*

In [ ]:
class Mux4to1 extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(4.W))
    val out = Output(UInt(1.W))
    val sel = Input(UInt(2.W))
  })
  
  when(io.sel===0.U) {
    io.out := io.in(0)
  }.elsewhen (io.sel===1.U) {
    io.out := io.in(1)
  }.elsewhen (io.sel===2.U) {
    io.out := io.in(2)
  }.otherwise {
    io.out := io.in(3)
  }
}

#### Τι πρέπει να προσέξετε στη δομή when..elsewhen..otherwise
α) Παρατηρήστε τη σύνταξη (προσέξτε ιδιαίτερα τις τελείες!):

~~~scala
when(συνθήκη) {
    
  }.elsewhen (συνθήκη) {
    
  }.otherwise {
    
  }
~~~

Δεν πρέπει να μπερεδεύετε το `when..elsewhen..otherwise` με το `if..else` της Scala. Το `if..else` εκτελείται άμεσα, ενώ το `when..elsewhen..otherwise` κατασκευάζει το κύκλωμα που όταν λειτουργεί διαλέγει πιθανές τιμές για κάποια σήματα.

β) Η Chisel διαθέτει τους εξής **τελεστές σύγκρισης**:

| Τελεστής | Έλεγχος |
| --- | --- |
| === | ισότητα |
| =/= | ανισότητα |
| >, >=, <, <= | σύγκριση |

Μπορείτε επίσης να συνδυάσετε συνθήκες π.χ. (προσέξτε πώς μπαίνουν οι παρενθέσεις):
~~~scala
when (io.x===0.U && io.y===1.U)  {
~~~

γ) Η δομή `when..elsewhen..otherwise` θα πρέπει να καλύπτει **κάθε πιθανό συνδυασμό**. Έτσι η χρήση του `otherwise` είναι **απαραίτητη**.

δ) Από την άλλη πλευρά, **δεν είναι απαραίτητη** η χρήση του `elsewhen`, παρά μόνο αν έχετε περισσότερες από δύο περιπτώσεις.

ε) Μπορείτε, αν απαιτείται, να χρησιμοποιήσετε nested δομές (π.χ. `when` μέσα σε άλλα `when`).

### 2.2.5 Περιγραφή συμπεριφοράς με τη δομή switch

Μια δεύτερη δομή της Chisel για την περιγραφή συμπεριφοράς είναι το `switch`.

Στο επόμενο κελί ο ίδιος πολυπλέκτης 4-σε-1 με τη δομή `switch`.

**Προσοχή! Η Chisel απαιτεί την αρχική τιμή (safeguard) του ``io.out`` (ίση με 0.U) για να μπορέσει να υλοποιήσει (elaborate) το κύκλωμα!**

*Εκτελέστε το επόμενο κελί που επανακαθορίζει ξανά το module `Mux4to1`. Στη συνέχεια τρέξτε πάλι το προηγούμενο test (ενότητα 2.2.3) για να βεβαιωθείτε ότι και αυτή η εκδοχή του πολυπλέκτη δουλεύει κατά το αναμενόμενο.*

In [ ]:
class Mux4to1 extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(4.W))
    val out = Output(UInt(1.W))
    val sel = Input(UInt(2.W))
  })
    
  io.out := 0.U   // safeguard
  switch(io.sel) {
    is(0.U) {  io.out := io.in(0) }
    is(1.U) {  io.out := io.in(1) }
    is(2.U) {  io.out := io.in(2) }
    is(3.U) {  io.out := io.in(3) }
  }
}

#### Τι πρέπει να προσέξετε στη δομή switch
Η Chisel (τουλάχιστον στην μέχρι τώρα έκδοσή της) δεν μπορεί να διακρίνει ότι τα 0.U,1.U,2.U,3.U είναι όλες οι περιπτώσεις τιμών για τα 2 bits του sel, συνεπώς δεν μπορεί να ξέρει αν έχετε καλύψει όλες τις περιπτώσεις για το `io.out`. Έτσι, οποιοδήποτε σήμα παίρνει τιμή μέσα σε κάποιο `is` θα πρέπει να έχει μια αρχική τιμή (safeguard) **πρίν και έξω** από το switch (είναι θέμα αντίστοιχο του `otherwise` της δομής `when`).

## 2.3 Άσκηση: Μονάδα λογικών πράξεων του ενός bit
Με βάση τα προηγούμενα, κατασκευάστε module `Logic` με τη δομή `when..elsewhen..otherwise` το οποίο θα έχει:

* Δύο εισόδους δεδομένων `a` και `b` εύρους 1 bit η καθεμία.
* Μία έξοδο δεδομένων `out` εύρους 1 bit.
* Μία είσοδο ελέγχου `s` εύρους 2 bits.

Το module `Logic` θα λειτουργεί σύμφωνα με τον παρακάτω πίνακα:

| s(1) | s(0) | out |
| ---- | ---- | --- |
| 0 | 0 | a AND b |
| 0 | 1 | a OR b |
| 1 | 0 | a XOR b |
| 1 | 1 | NOT a |


In [ ]:
class Logic extends Module {
  val io = IO(new Bundle {
      
    // ..ορίστε τις εισόδους και εξόδους εδώ..
      
  })
  
  // ..καθορίστε τη ζητούμενη λειτουργικότητα με τη δομή when εδώ..
    
}

Ελέγξτε την ορθότητα της υλοποίησής σας εκτελώντας το επόμενο κελί:

In [ ]:
try {
  test(new Logic()) { c =>
    for (aval <- 0 to 1) {  // για τις 2 πιθανές τιμές του io.a
      c.io.a.poke(aval.U)
      for (bval <- 0 to 1) {  // για τις 2 πιθανές τιμές του io.b
        c.io.b.poke(bval.U)
        // έλεγχος εξόδων κάνοντας τις αντίστοιχες πράξεις με τη Scala για επαλήθευση αποτελέσματος
        c.io.s.poke(0.U)  // AND
        c.io.out.expect((aval & bval).U)
        c.io.s.poke(1.U)  // OR
        c.io.out.expect((aval | bval).U)
        c.io.s.poke(2.U)  // XOR
        c.io.out.expect((aval ^ bval).U)
        c.io.s.poke(3.U)  // +
        val result = (~aval) & 0x1  // αφαίρεση όλων των 1 εκτός του τελαυταίου από το αποτέλεσμα της Scala
        c.io.out.expect(result.U)
      }
    }
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

Περιγράψτε στη συνέχεια το ίδιο module `Logic` με τη βοήθεια της δομής `switch`:

In [ ]:
class Logic extends Module {
  val io = IO(new Bundle {
      
    // ..ορίστε τις εισόδους και εξόδους εδώ..
      
  })
  
  // ..καθορίστε τη ζητούμενη λειτουργικότητα με τη δομή switch εδώ..
    
}

Δοκιμάστε με το προηγούμενο `test` την ορθότητα και αυτού του κυκλώματος.

## Τελειώνοντας...
Μπορείτε να κατεβάσετε και να πάρετε μαζί σας το σημερινό notebook για να το έχετε στο αρχείο σας.

**Δεν ανήκει στα παραδοτέα του εργαστηρίου.**